In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('../data/delivery_data.csv')

# Drop rows with missing source/destination names
df = df.dropna(subset=['source_name', 'destination_name'])

# Remove impossible segment_factor values (negative = data error)
df = df[df['segment_factor'] > 0]

print("Cleaned dataset shape:", df.shape)
print("Removed rows:", 144867 - df.shape[0])

Cleaned dataset shape: (141725, 24)
Removed rows: 3142


In [2]:
# Build directed graph
# Each unique route (source -> destination) becomes an edge
# Edge weight = median segment_factor (delay ratio) for that corridor

G = nx.DiGraph()  # Directed graph - direction matters (A->B is different from B->A)

# Group trips by corridor (source -> destination pair)
corridor_stats = df.groupby(['source_center', 'destination_center']).agg(
    median_delay = ('segment_factor', 'median'),
    trip_count = ('trip_uuid', 'count'),
    median_actual_time = ('actual_time', 'median'),
    median_osrm_time = ('osrm_time', 'median'),
    median_distance = ('actual_distance_to_destination', 'median')
).reset_index()

# Add edges to graph
for _, row in corridor_stats.iterrows():
    G.add_edge(
        row['source_center'],
        row['destination_center'],
        weight=row['median_delay'],
        trip_count=row['trip_count'],
        actual_time=row['median_actual_time'],
        osrm_time=row['median_osrm_time'],
        distance=row['median_distance']
    )

print(f"Graph created!")
print(f"Number of nodes (hubs): {G.number_of_nodes()}")
print(f"Number of edges (corridors): {G.number_of_edges()}")

Graph created!
Number of nodes (hubs): 1641
Number of edges (corridors): 2741


In [3]:
# Basic graph properties
print("=== Graph Summary ===")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Is directed: {G.is_directed()}")

# Degree analysis - how many connections does each hub have?
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

print(f"\nTop 10 Hubs by IN-degree (most packages arriving):")
top_in = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
for hub, deg in top_in:
    print(f"  {hub}: {deg} incoming routes")

print(f"\nTop 10 Hubs by OUT-degree (most packages leaving):")
top_out = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
for hub, deg in top_out:
    print(f"  {hub}: {deg} outgoing routes")

=== Graph Summary ===
Nodes: 1641
Edges: 2741
Is directed: True

Top 10 Hubs by IN-degree (most packages arriving):
  IND000000ACB: 45 incoming routes
  IND562132AAA: 36 incoming routes
  IND160002AAC: 32 incoming routes
  IND501359AAE: 30 incoming routes
  IND421302AAG: 29 incoming routes
  IND712311AAA: 24 incoming routes
  IND411033AAA: 23 incoming routes
  IND131028AAB: 20 incoming routes
  IND110037AAM: 20 incoming routes
  IND600056AAB: 18 incoming routes

Top 10 Hubs by OUT-degree (most packages leaving):
  IND000000ACB: 47 outgoing routes
  IND562132AAA: 35 outgoing routes
  IND160002AAC: 29 outgoing routes
  IND421302AAG: 29 outgoing routes
  IND501359AAE: 27 outgoing routes
  IND712311AAA: 22 outgoing routes
  IND110037AAM: 22 outgoing routes
  IND411033AAA: 20 outgoing routes
  IND131028AAB: 19 outgoing routes
  IND600056AAB: 18 outgoing routes


In [4]:
# Find chronically delayed corridors (actual time > OSRM by more than 20%)
# segment_factor > 1.2 means actual took 20% longer than predicted

delayed_corridors = corridor_stats[corridor_stats['median_delay'] > 1.2].copy()
delayed_corridors = delayed_corridors.sort_values('median_delay', ascending=False)

print(f"Total corridors: {len(corridor_stats)}")
print(f"Chronically delayed corridors (>20% delay): {len(delayed_corridors)}")
print(f"Percentage: {len(delayed_corridors)/len(corridor_stats)*100:.1f}%")

print(f"\nTop 15 Most Delayed Corridors:")
print(delayed_corridors[['source_center','destination_center',
                          'median_delay','trip_count',
                          'median_actual_time','median_osrm_time']].head(15).to_string())

Total corridors: 2741
Chronically delayed corridors (>20% delay): 2581
Percentage: 94.2%

Top 15 Most Delayed Corridors:
     source_center destination_center  median_delay  trip_count  median_actual_time  median_osrm_time
1774  IND571105AAA       IND571114AAA     81.700000           2               116.0              24.5
446   IND208012AAA       IND209304AAA     35.000000          33               465.0              15.0
2519  IND785634AAA       IND785001AAA     31.200000           3               468.0              16.0
2672  IND842003AAB       IND482002AAA     23.187500           4               185.5               8.0
1187  IND424304AAC       IND424006AAA     22.313663           2               926.0              41.5
1192  IND425405AAA       IND424006AAA     21.185587          18              1017.0              46.0
2311  IND722140AAA       IND723130AAA     19.130435           1              1320.0              69.0
144   IND121002AAA       IND121004AAB     19.083333           1

In [5]:
# Save corridor stats for later use
corridor_stats.to_csv('../data/corridor_stats.csv', index=False)

# Save graph
nx.write_gexf(G, '../data/logistics_graph.gexf')

print("Corridor stats saved to data/corridor_stats.csv")
print("Graph saved to data/logistics_graph.gexf")
print(f"\nGraph ready for bottleneck analysis!")
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

Corridor stats saved to data/corridor_stats.csv
Graph saved to data/logistics_graph.gexf

Graph ready for bottleneck analysis!
Nodes: 1641, Edges: 2741
